# ❄️ Introducción al modelo SARIMA


El modelo **SARIMA** (AutoRegresivo Integrado de Media Móvil Estacional) extiende el modelo ARIMA incorporando efectos estacionales. Es útil para series de tiempo que muestran patrones repetitivos a intervalos regulares.

SARIMA se representa como:

**SARIMA(p, d, q)(P, D, Q, s)**
Donde:
- **p, d, q**: son los parámetros del modelo ARIMA (no estacional).
- **P, D, Q**: son los parámetros estacionales (orden autorregresivo, diferenciación y media móvil).
- **s**: es la longitud del período estacional (por ejemplo, 12 para datos mensuales con patrón anual).

### 1️⃣ 📥 Carga de datos


In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv")
df.columns = ["Fecha", "Pasajeros"]
df["Fecha"] = pd.to_datetime(df["Fecha"])
df.set_index("Fecha", inplace=True)
df.head()

### 2️⃣ 📊 Visualización de la serie


In [ ]:
import matplotlib.pyplot as plt
df.plot(figsize=(10, 4), title="Número de pasajeros mensuales", ylabel="Pasajeros")
plt.grid()
plt.show()

### 3️⃣ 🔁 Detección de estacionalidad


Para usar un modelo SARIMA, primero debemos confirmar que la serie presenta **estacionalidad**. Esto puede hacerse visualmente o mediante la **descomposición estacional**.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
descomposicion = seasonal_decompose(df["Pasajeros"], model="additive", period=12)
descomposicion.plot()
plt.show()

La descomposición muestra una clara estacionalidad anual (cada 12 meses), lo que indica que un modelo SARIMA es apropiado.

### 4️⃣ ⚖️ Estacionariedad: determinación de d y D


In [ ]:
from statsmodels.tsa.stattools import adfuller
# Prueba Dickey-Fuller sin diferenciación
adf_orig = adfuller(df["Pasajeros"])
print(f"Valor p original: {adf_orig[1]}")
# Diferencia no estacional
df_diff = df.diff().dropna()
adf_diff = adfuller(df_diff["Pasajeros"])
print(f"Valor p tras d=1: {adf_diff[1]}")
# Diferencia estacional
df_seasonal_diff = df.diff(12).dropna()
adf_seasonal = adfuller(df_seasonal_diff["Pasajeros"])
print(f"Valor p tras D=1: {adf_seasonal[1] < 0.05}")


Después de aplicar una diferencia no estacional y una estacional (rezago 12), la serie se vuelve estacionaria. Por tanto, tomamos: **d = 1**, **D = 1**, **s = 12**.

### 5️⃣ 📉 ACF y PACF para determinar p, q, P, Q


Aplicamos ACF y PACF a la serie diferenciada para estimar los parámetros del modelo no estacional y estacional.

In [ ]:
plot_acf = plot_pacf(df.diff().dropna())

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
df_full_diff = df.diff(12).dropna()
plot_acf(df_full_diff, lags=36)
plt.show()
plot_pacf(df_full_diff, lags=36)
plt.show()

- A partir de los gráficos:
  - **p** y **q** se estiman observando cortes tempranos en PACF y ACF.
  - **P** y **Q** se estiman observando lags múltiples de 12.

In [ ]:
p = 1
d = 1
q = 1
s = 12
P = 2
D = 1
Q = 1

In [ ]:
serie = df['Pasajeros']

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
SARIMA_model = SARIMAX(serie, order = (p, d, q), seasonal_order = (P, D, Q, s))

In [ ]:
resultado = SARIMA_model.fit()

In [ ]:
resultado.predict(start = '1961-01', stop = '1962-01')